In [4]:
INCLUDE_EMPTY = False          # drop cases with no tumor & no pancreas
REQUIRE_PANCREAS = True        # only include pseudo if pancreas_vox > 0
KEEP_TUMOR_POSITIVES = True    # include tumor-positive pseudo

# === paths/env ===
import os, json, shutil, re
from pathlib import Path
import pandas as pd

os.environ['nnUNet_raw']          = r'E:\nnUNet_raw'
os.environ['nnUNet_preprocessed'] = r'E:\nnUNet_preprocessed'
os.environ['nnUNet_results']      = r'E:\nnUNet_results'

ORIG_DS    = Path(r"E:\nnUNet_raw\Dataset110_PANTHER_T1")   # GT (multi-class)
UNLAB_NII  = Path(r"E:\499\Data\task 1\ImagesTr_unlabeled_nii")
PSEUDO_DIR = Path(r"E:\nnUNet_raw\Dataset110_PANTHER_T1\LabelsPseudo_multiclass")
QC_CSV     = PSEUDO_DIR/"multiclass_qc.csv"

DS_ID   = 114
DS_NAME = f"Dataset{DS_ID:03d}_PANTHER_T1_multiclass_GT_pseudo"
RAW_DS  = Path(os.environ['nnUNet_raw'])/DS_NAME
IMGS_TR = RAW_DS/'imagesTr'
LABS_TR = RAW_DS/'labelsTr'

# clean / create
if RAW_DS.exists(): shutil.rmtree(RAW_DS)
IMGS_TR.mkdir(parents=True, exist_ok=True)
LABS_TR.mkdir(parents=True, exist_ok=True)

def niigz_stem(p: Path) -> str:
    return p.name[:-7] if p.name.endswith(".nii.gz") else p.stem

def strip_modality(stem: str) -> str:
    m = re.match(r"^(.*)_0\d{3}$", stem)
    return m.group(1) if m else stem

def find_unlab_img(stem: str) -> Path|None:
    for cand in [
        UNLAB_NII/f"{stem}.nii.gz",
        UNLAB_NII/f"{stem}_0000.nii.gz",
        UNLAB_NII/f"{strip_modality(stem)}_0000.nii.gz",
    ]:
        if cand.exists(): return cand
    return None

pairs = []
# 1) add ALL GT (multi-class)
gt_imgs = ORIG_DS/'imagesTr'
gt_lbls = ORIG_DS/'labelsTr'
added_gt = 0
for lbl in sorted(gt_lbls.glob("*.nii.gz")):
    stem = niigz_stem(lbl)
    img  = gt_imgs/f"{stem}_0000.nii.gz"
    if not img.exists():
        alt = gt_imgs/f"{stem}.nii.gz"
        img = alt if alt.exists() else img
    if not img.exists():
        print(" GT image missing:", stem); continue
    shutil.copy2(img, IMGS_TR/img.name)
    shutil.copy2(lbl, LABS_TR/f"{stem}.nii.gz")
    pairs.append({"image": f"./imagesTr/{img.name}", "label": f"./labelsTr/{stem}.nii.gz"})
    added_gt += 1

# 2) add filtered pseudo (positives AND pancreas-only negatives)
qc = pd.read_csv(QC_CSV)
qc["stem"] = qc["case"].str.replace(".nii.gz","",regex=False)

keep = qc.copy()
if not INCLUDE_EMPTY:
    keep = keep[~((keep.tumor_vox==0) & (keep.pancreas_vox==0))]
if REQUIRE_PANCREAS:
    keep = keep[keep.pancreas_vox > 0]
if KEEP_TUMOR_POSITIVES:
    # union of tumor+ and pancreas-only negatives is already implied by the two filters above
    pass

added_pseudo = 0; skipped_dupe = 0; missing_imgs = []
kept_pos = 0; kept_neg_pancreas = 0
for _, r in keep.iterrows():
    stem = r["stem"]
    is_pos = r["tumor_vox"] > 0
    if (LABS_TR/f"{stem}.nii.gz").exists():  # prefer GT
        skipped_dupe += 1
        continue
    img = find_unlab_img(stem)
    if img is None:
        missing_imgs.append(stem); continue
    out_img = IMGS_TR/img.name
    out_lbl = LABS_TR/f"{stem}.nii.gz"
    shutil.copy2(img, out_img)
    shutil.copy2(PSEUDO_DIR/f"{stem}.nii.gz", out_lbl)
    pairs.append({"image": f"./imagesTr/{out_img.name}", "label": f"./labelsTr/{out_lbl.name}"})
    added_pseudo += 1
    kept_pos += int(is_pos)
    kept_neg_pancreas += int(not is_pos)

# write dataset.json & manifest
dataset = {
    "name": DS_NAME,
    "description": "GT multi-class + multiclass pseudo (MV; tumor gated by pancreas). Pseudo: tumor+ and pancreas-only negatives; empties dropped.",
    "tensorImageSize": "3D",
    "modality": {"0": "MRI"},
    "labels": {"0": "background", "1": "tumor", "2": "pancreas"},
    "numTraining": len(pairs),
    "numTest": 0,
    "training": pairs,
    "test": []
}
with open(RAW_DS/'dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)
pd.DataFrame(pairs).to_csv(RAW_DS/"training_manifest.csv", index=False)

print(f"\nBuilt {DS_NAME}")
print("Added GT:", added_gt)
print(f"Added pseudo total: {added_pseudo}  | tumor+: {kept_pos}  | pancreas-only negatives: {kept_neg_pancreas}")
if missing_imgs:
    print("Missing unlabeled images for:", len(missing_imgs), "e.g.", missing_imgs[:10])
print("imagesTr:", len(list(IMGS_TR.glob('*.nii.gz'))), "| labelsTr:", len(list(LABS_TR.glob('*.nii.gz'))))
print("Manifest:", RAW_DS/"training_manifest.csv")



Built Dataset114_PANTHER_T1_multiclass_GT_pseudo
Added GT: 92
Added pseudo total: 329  | tumor+: 192  | pancreas-only negatives: 137
imagesTr: 421 | labelsTr: 421
Manifest: E:\nnUNet_raw\Dataset114_PANTHER_T1_multiclass_GT_pseudo\training_manifest.csv


In [1]:
import sys, nnunetv2, inspect
print("Python:", sys.executable)
print("nnunetv2:", inspect.getfile(nnunetv2))


Python: C:\Users\Redwan\anaconda3\envs\panther-ens\python.exe
nnunetv2: C:\Users\Redwan\nnUNet\nnunetv2\__init__.py


In [1]:
import os
os.environ['nnUNet_raw']          = r'E:\nnUNet_raw'
os.environ['nnUNet_preprocessed'] = r'E:\nnUNet_preprocessed'
os.environ['nnUNet_results']      = r'E:\nnUNet_results'
os.environ['nnUNet_n_proc_DA']    = '12'   # tune to your CPU
os.environ['nnUNet_max_num_threads'] = '12'
DATASET_ID = 114


In [3]:
from pathlib import Path
RAW = Path(os.environ['nnUNet_raw'])
cands = list(RAW.glob(f'Dataset{DATASET_ID}_*'))
print("Found:", cands)
assert cands, f"Dataset{DATASET_ID}_* not found in {RAW}"
DS = cands[0]
for p in [DS/'imagesTr', DS/'labelsTr', DS/'dataset.json']:
    print(p.exists(), p)


Found: [WindowsPath('E:/nnUNet_raw/Dataset114_PANTHER_T1_multiclass_GT_pseudo')]
True E:\nnUNet_raw\Dataset114_PANTHER_T1_multiclass_GT_pseudo\imagesTr
True E:\nnUNet_raw\Dataset114_PANTHER_T1_multiclass_GT_pseudo\labelsTr
True E:\nnUNet_raw\Dataset114_PANTHER_T1_multiclass_GT_pseudo\dataset.json


In [4]:
import json, os
from pathlib import Path

RAW_DS = Path(r"E:\nnUNet_raw\Dataset114_PANTHER_T1_multiclass_GT_pseudo")
imagesTr = RAW_DS / "imagesTr"
labelsTr = RAW_DS / "labelsTr"

assert imagesTr.exists() and labelsTr.exists(), "imagesTr / labelsTr folders missing"

# collect pairs: image 'CASE_0000.nii.gz' + label 'CASE.nii.gz'
pairs = []
missing = []
for img in sorted(imagesTr.glob("*_0000.nii.gz")):
    stem = img.name[:-12]          # drop '_0000.nii.gz'
    lbl  = labelsTr / f"{stem}.nii.gz"
    if lbl.exists():
        pairs.append({"image": f"./imagesTr/{img.name}", "label": f"./labelsTr/{lbl.name}"})
    else:
        missing.append(stem)

print(f"Found {len(pairs)} pairs. Missing labels for {len(missing)} images.")
if missing: print("Examples:", missing[:10])

dataset = {
    "dataset_name": "PANTHER_T1_multiclass_GT_pseudo",
    "channel_names": {"0": "MRI"},
    "labels": {"background": 0, "tumor": 1, "pancreas": 2},
    "numTraining": len(pairs),
    "file_ending": ".nii.gz",
    "overwrite_image_reader_writer": "SimpleITKIO",
    "training": pairs,
    "test": []
}

with open(RAW_DS / "dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

print(" dataset.json written to", RAW_DS)


Found 421 pairs. Missing labels for 0 images.
 dataset.json written to E:\nnUNet_raw\Dataset114_PANTHER_T1_multiclass_GT_pseudo


In [9]:
import os, sys
# nnU-Net paths
os.environ['nnUNet_raw']          = r'E:\nnUNet_raw'
os.environ['nnUNet_preprocessed'] = r'E:\nnUNet_preprocessed'
os.environ['nnUNet_results']      = r'E:\nnUNet_results'
os.environ['nnUNet_n_proc_DA']    = '12'   # tune to your CPU if you like

DATASET_ID = 114

from nnunetv2.experiment_planning.plan_and_preprocess_entrypoints import (
    extract_fingerprint_entry,
    plan_and_preprocess_entry,
)

def run_extract(ds):
    sys.argv = ["nnUNetv2_extract_fingerprint", "-d", str(ds)]
    print(">>>", " ".join(sys.argv))
    extract_fingerprint_entry()

def run_plan(ds):
    sys.argv = ["nnUNetv2_plan_and_preprocess", "-d", str(ds)]
    print(">>>", " ".join(sys.argv))
    plan_and_preprocess_entry()

run_extract(DATASET_ID)
run_plan(DATASET_ID)


>>> nnUNetv2_extract_fingerprint -d 114


RuntimeError: Could not find a dataset with the ID 114. Make sure the requested dataset ID exists and that nnU-Net knows where raw and preprocessed data are located (see Documentation - Installation). Here are your currently defined folders:
nnUNet_preprocessed=E:\nnUNet_preprocessed
nnUNet_results=E:\nnUNet_results
nnUNet_raw=E:\nnUNet_raw
If something is not right, adapt your environment variables.

In [15]:
import os, sys
from nnunetv2.run.run_training import run_training_entry

# ---- nnU-Net paths (your paths are correct) ----
os.environ['nnUNet_raw']          = r'E:\nnUNet_raw'
os.environ['nnUNet_preprocessed'] = r'E:\nnUNet_preprocessed'
os.environ['nnUNet_results']      = r'E:\nnUNet_results'

# CPU workers & which GPU to use
os.environ['nnUNet_n_proc_DA']    = '12'   # tune for your CPU
os.environ['CUDA_VISIBLE_DEVICES']= '0'    # pick GPU 0

DATASET_ID = 114
CKPT = r"E:\nnUNet_results\Dataset110_PANTHER_T1\nnUNetTrainer__nnUNetPlans__3d_fullres\fold_0\checkpoint_best.pth"


sys.argv = [
    "nnUNetv2_train",
    str(DATASET_ID), "3d_fullres", "0",        # dataset, configuration, fold
    "-tr", "nnUNetTrainer",
    "-p",  "nnUNetPlans",
    "-pretrained_weights", CKPT,
    "-num_gpus", "1",                          # optional, safe on single-GPU
    # "--npz",                                  # optional: save softmax
    # "--disable_checkpointing",                # optional if disk is tight
]
run_training_entry()



############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

################### Loading pretrained weights from file  E:\nnUNet_results\Dataset110_PANTHER_T1\nnUNetTrainer__nnUNetPlans__3d_fullres\fold_0\checkpoint_best.pth ###################
Below is the list of overlapping blocks in pretrained model and nnUNet architecture:
encoder.stages.0.0.convs.0.

RuntimeError: One or more background workers are no longer alive. Exiting. Please check the print statements above for the actual error message

In [1]:
import os, glob, json, sys, subprocess
from pathlib import Path

# Make sure env vars are set in THIS kernel
os.environ['nnUNet_raw'] = r'E:\nnUNet_raw'
os.environ['nnUNet_preprocessed'] = r'E:\nnUNet_preprocessed'
os.environ['nnUNet_results'] = r'E:\nnUNet_results'
print("nnUNet_raw        =", os.environ['nnUNet_raw'])
print("nnUNet_preprocessed =", os.environ['nnUNet_preprocessed'])
print("nnUNet_results    =", os.environ['nnUNet_results'])

RAW = os.environ['nnUNet_raw']
PRE = os.environ['nnUNet_preprocessed']

raw114 = glob.glob(os.path.join(RAW, "Dataset114_*"))
pre114 = glob.glob(os.path.join(PRE, "Dataset114_*"))

print("\nRaw 114 folders:", raw114)
print("Pre 114 folders:", pre114)

# Check dataset.json in raw
for p in raw114:
    dj = os.path.join(p, "dataset.json")
    print("dataset.json exists in", p, "->", os.path.isfile(dj))


nnUNet_raw        = E:\nnUNet_raw
nnUNet_preprocessed = E:\nnUNet_preprocessed
nnUNet_results    = E:\nnUNet_results

Raw 114 folders: ['E:\\nnUNet_raw\\Dataset114_PANTHER_T1_multiclass_GT_pseudo']
Pre 114 folders: ['E:\\nnUNet_preprocessed\\Dataset114_PANTHER_T1_multiclass_GT_pseudo']
dataset.json exists in E:\nnUNet_raw\Dataset114_PANTHER_T1_multiclass_GT_pseudo -> True


In [2]:
import os, sys
from nnunetv2.run.run_training import run_training_entry

# ---- nnU-Net paths (your paths are correct) ----
os.environ['nnUNet_raw']          = r'E:\nnUNet_raw'
os.environ['nnUNet_preprocessed'] = r'E:\nnUNet_preprocessed'
os.environ['nnUNet_results']      = r'E:\nnUNet_results'

# CPU workers & which GPU to use

os.environ['CUDA_VISIBLE_DEVICES']= '0'    # pick GPU 0

DATASET_ID = 114
CKPT = r"E:\nnUNet_results\Dataset110_PANTHER_T1\nnUNetTrainer__nnUNetPlans__3d_fullres\fold_1\checkpoint_best.pth"


sys.argv = [
    "nnUNetv2_train",
    str(DATASET_ID), "3d_fullres", "1",        # dataset, configuration, fold
    "-tr", "nnUNetTrainer",
    "-p",  "nnUNetPlans",
    "-pretrained_weights", CKPT,
    "-num_gpus", "1",                          # optional, safe on single-GPU
    # "--npz",                                  # optional: save softmax
    # "--disable_checkpointing",                # optional if disk is tight
]
run_training_entry()



############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

################### Loading pretrained weights from file  E:\nnUNet_results\Dataset110_PANTHER_T1\nnUNetTrainer__nnUNetPlans__3d_fullres\fold_1\checkpoint_best.pth ###################
Below is the list of overlapping blocks in pretrained model and nnUNet architecture:
encoder.stages.0.0.convs.0.

RuntimeError: One or more background workers are no longer alive. Exiting. Please check the print statements above for the actual error message

In [2]:
import os, sys
from nnunetv2.run.run_training import run_training_entry

# ---- nnU-Net paths (your paths are correct) ----
os.environ['nnUNet_raw']          = r'E:\nnUNet_raw'
os.environ['nnUNet_preprocessed'] = r'E:\nnUNet_preprocessed'
os.environ['nnUNet_results']      = r'E:\nnUNet_results'

# CPU workers & which GPU to use

os.environ['CUDA_VISIBLE_DEVICES']= '0'    # pick GPU 0

DATASET_ID = 114
CKPT = r"E:\nnUNet_results\Dataset110_PANTHER_T1\nnUNetTrainer__nnUNetPlans__3d_fullres\fold_2\checkpoint_best.pth"


sys.argv = [
    "nnUNetv2_train",
    str(DATASET_ID), "3d_fullres", "2",        # dataset, configuration, fold
    "-tr", "nnUNetTrainer",
    "-p",  "nnUNetPlans",
    "-pretrained_weights", CKPT,
    "-num_gpus", "1",                          # optional, safe on single-GPU
    # "--npz",                                  # optional: save softmax
    # "--disable_checkpointing",                # optional if disk is tight
]
run_training_entry()



############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

################### Loading pretrained weights from file  E:\nnUNet_results\Dataset110_PANTHER_T1\nnUNetTrainer__nnUNetPlans__3d_fullres\fold_2\checkpoint_best.pth ###################
Below is the list of overlapping blocks in pretrained model and nnUNet architecture:
encoder.stages.0.0.convs.0.


KeyboardInterrupt

